# ***Project Title: Customer Transaction Prediction***

# ***1.Problem Statement***

Banks have a large number of customers, but not all customers perform transactions regularly. Identifying customers who are likely to make transactions in the future is important for improving customer engagement and business growth. This project uses historical customer data to analyze customer behavior and develop a predictive model that can identify which customers are likely to make transactions in the future.

### Objective
1) Prepare a complete data analysis report on the given customer transaction dataset.
2) Create a predictive model that helps the bank identify which customers are likely to make transactions in the future.

### Target:
* 0 = No Transaction
* 1 = Transaction

## ***Domain Analysis***
**Domain:** Banking

# ***2.Import Libraries***

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

# ***3.Load Dataset***

In [ ]:
df = pd.read_csv("train_sample.csv")

In [ ]:
df.head()

In [ ]:
df.shape

# ***4.Dataset Overview***

In [ ]:
df.info()

In [ ]:
df.describe()

# ***5.Missing Values & Duplicates***

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().sum().sum()

In [ ]:
df.duplicated().sum()

# ***6.Target Variable Analysis***

In [ ]:
df["target"].value_counts()

In [ ]:
sns.countplot(x='target',data=df)
plt.title("Target Distribution")
plt.show()

# ***7.Percentage Distribution***

In [ ]:
target_percent = df["target"].value_counts(normalize=True)*100
print(target_percent)

# ***8.Remove ID Column***

In [ ]:
df.drop("ID_code",axis=1,inplace=True)

# ***9.Feature and Target Separation***

In [ ]:
X = df.drop("target",axis=1)
y = df["target"]

# ***10.Train Test Split***

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.20,random_state=42,
    stratify=y)

# ***11.Feature Scaling***

In [ ]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ***12.Smote***

In [ ]:
smote = SMOTE(random_state=42)

X_train, y_train = smote.fit_resample(X_train, y_train)

# ***13.Model Building***

## A.Logistic Regression

In [ ]:
lr = LogisticRegression()
lr.fit(X_train,y_train)
pred_lr = lr.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test,pred_lr))
print("Precision:", precision_score(y_test,pred_lr))
print("Recall:", recall_score(y_test,pred_lr))
print("F1:",f1_score(y_test,pred_lr))
print("ROC AUC:", roc_auc_score(y_test,pred_lr))

## 2.Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train,y_train)
pred_dt = dt.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test,pred_dt))
print("Precision:", precision_score(y_test,pred_dt))
print("Recall:", recall_score(y_test,pred_dt))
print("F1:",f1_score(y_test,pred_dt))
print("ROC AUC:", roc_auc_score(y_test,pred_dt))

## 3.Random Forest

In [ ]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train,y_train)
pred_rf = rf.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test,pred_rf))
print("Precision:", precision_score(y_test,pred_rf))
print("Recall:", recall_score(y_test,pred_rf))
print("F1:",f1_score(y_test,pred_rf))
print("ROC AUC:", roc_auc_score(y_test,pred_rf))

## 4.Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier()
gb.fit(X_train,y_train)
pred_gb = gb.predict(X_test)

In [ ]:
print("Accuracy:", accuracy_score(y_test,pred_gb))
print("Precision:", precision_score(y_test,pred_gb))
print("Recall:", recall_score(y_test,pred_gb))
print("F1:",f1_score(y_test,pred_gb))
print("ROC AUC:", roc_auc_score(y_test,pred_gb))

# ***14.Model Comparison Table***

In [ ]:
results = pd.DataFrame({
    'Model':["Logidtic Regression",
             "Decision Tree",
             "Random Forest",
             "Gradient Boosting"],

    "Accuracy":[accuracy_score(y_test,pred_lr),
                accuracy_score(y_test,pred_dt),
                accuracy_score(y_test,pred_rf),
                accuracy_score(y_test,pred_gb)],

    "F1 Score":[f1_score(y_test,pred_lr),
                f1_score(y_test,pred_dt),
                f1_score(y_test,pred_rf),
                f1_score(y_test,pred_gb)],

    "ROC AUC":[roc_auc_score(y_test,pred_lr),
               roc_auc_score(y_test,pred_dt),
               roc_auc_score(y_test,pred_rf),
               roc_auc_score(y_test,pred_gb)]
})

results

# ***15.Confusion Martix***

In [ ]:
cm = confusion_matrix(y_test,pred_rf)

plt.figure(figsize=(6,4))
sns.heatmap(cm,annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

# ***16.Classification Report***

In [ ]:
print(classification_report(y_test, pred_rf))

# ***17.Hyperparameter Tuning***

In [ ]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators':[100,200],
        'max_depth':[5,10]
    },
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=2
)

grid.fit(X_train,y_train)

print(grid.best_params_)

# ***18.Final Model***

In [ ]:
best_rf = RandomForestClassifier(
n_estimators=200,
max_depth=10,
random_state=42
)

best_rf.fit(X_train,y_train)

final_pred = best_rf.predict(X_test)

# ***19.Cross Validation***

In [ ]:
cv_scores = cross_val_score(
    best_rf,X_train,y_train,cv=5,scoring="roc_auc")

print(cv_scores)

print("average ROC AUC:",cv_scores.mean())

# ***20.Feature Importance***

In [ ]:
importance = best_rf.feature_importances_

features = X.columns

feature_importance = pd.DataFrame({
    'Feature':features,
    'Importance':importance
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

feature_importance.head(20)

# ***21.Business Insights***

1) 90% of customers do not perform transaction.
2) Only a small group of customers actively respond to banking offers.
3) Only 10% of customers are likely to make transactions.
4) These customers are important for increasing bank revenue.
5) The model helps identify customers who are more likely to respond.
6) The model helps to find customers interested in banking products.
7) The dataset is highly imbalanced.
8) The bank can reduce marketing costs by focusing on likely customers.
9) Customer behavior can be predicted using historical data.
10) Machine learning helps improve decision-making.

# ***22.Challenges Faced***

**1. Anonymous Features**

* Challenge: The dataset contains 200 features with names like var_0, var_1, etc., making it difficult to understand their real meaning.

* Solution: Statistical analysis, correlation analysis, and feature importance techniques were used to understand the impact of features.

**2. Imbalanced Dataset**

* Challenge: Around 90% of customers belong to class 0 and only 10% belong to class 1, which can bias the model.

* Solution: Evaluation metrics such as Recall, F1-Score, and ROC-AUC were used, and balancing techniques like SMOTE can be applied.

**3. High Dimensional Data**

* Challenge: The dataset contains 200 features, increasing model complexity and training time.

* Solution: Feature importance analysis was performed to identify the most influential features.

**4. Risk of Overfitting**

* Challenge: Complex models may memorize training data and perform poorly on unseen data.

* Solution: Train-test split, cross-validation, and hyperparameter tuning were used to improve generalization.

**5. Model Selection**

* Challenge: Multiple machine learning algorithms produced different results, making model selection difficult.

* Solution: Models were compared using Accuracy, Precision, Recall, F1-Score, and ROC-AUC Score to select the best-performing model.

**6. Feature Scaling Requirement**

* Challenge: Features had different numerical ranges, which could affect model performance.

* Solution: StandardScaler was applied to normalize feature values before training.

**7. Identifying Minority-Class Customers**

* Challenge: Customers who perform transactions represent only a small portion of the dataset.

* Solution: Focus was placed on Recall and ROC-AUC Score to improve identification of potential transaction customers.


# ***23.Conclusion***

The goal of this project was to predict whether a customer will make a transaction or not. Different machine learning models were built and compared to find the best-performing model.

The dataset was analyzed, cleaned, and prepared before model training. The selected model showed good performance in identifying potential customers.

This model can help banks target the right customers, reduce marketing costs, improve campaign success rates, and increase business profit. Therefore, machine learning can be effectively used to support customer transaction prediction and improve business decision-making.